# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [5]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [6]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [7]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [8]:
# TODO: Load environment variables
load_dotenv("/workspace/Code/project/starter/.env", override=True)

True

### VectorDB Instance

In [9]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chromadb")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [10]:
import os
print("cwd:", os.getcwd())
print(".env here?", os.path.exists(".env"))
print("OPENAI:", os.getenv("OPENAI_API_KEY") is not None)
print("CHROMA:", os.getenv("CHROMA_OPENAI_API_KEY") is not None)

cwd: /workspace/Code/project/starter
.env here? True
OPENAI: True
CHROMA: True


### Collection

In [11]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY"),
    api_base="https://openai.vocareum.com/v1",
    model_name="text-embedding-3-small",
)

In [12]:
# TODO: Create a collection
# Choose any name you want
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Add documents

In [13]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


In [16]:
print(sorted(m["Name"] for m in collection.get()["metadatas"]))

for q in ["What platform was Pokémon Red launched on?",
          "What is Rockstar Games working on right now?"]:
    print("\nQ:", q)
    results = collection.query(query_texts=[q], n_results=3)
    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        print(f"{dist:.3f}  {doc}")

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


['Gran Turismo', 'Gran Turismo 5', 'Grand Theft Auto: San Andreas', 'Halo Infinite', 'Kinect Adventures!', 'Mario Kart 8 Deluxe', "Marvel's Spider-Man", "Marvel's Spider-Man 2", 'Minecraft', 'Pokémon Gold and Silver', 'Pokémon Ruby and Sapphire', 'Super Mario 64', 'Super Mario World', 'Super Smash Bros. Melee', 'Wii Sports']

Q: What platform was Pokémon Red launched on?
0.999  [Game Boy Color] Pokémon Gold and Silver (1999) - Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.
1.109  [Game Boy Advance] Pokémon Ruby and Sapphire (2002) - Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.
1.379  [Super Nintendo Entertainment System (SNES)] Super Mario World (1990) - A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.

Q: What is Rockstar Games working on right now?
1.216  [PlayStation 2] Grand Theft Auto: San Andreas (2004) - An expansive open-wo

In [15]:
results = collection.query(
    query_texts=["Which company developed Gran Turismo?"],
    n_results=3
)
for doc, dist in zip(results["documents"][0], results["distances"][0]):
    print(f"{dist:.3f}  {doc}")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


0.745  [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.
0.792  [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.
1.323  [PlayStation 2] Grand Theft Auto: San Andreas (2004) - An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson.


In [14]:
print("count:", collection.count())
collection.peek(2)

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


count: 15


{'ids': ['001', '002'],
 'embeddings': array([[-0.04354858,  0.02105713, -0.04354858, ...,  0.02972412,
          0.00756073, -0.0089798 ],
        [-0.01094055, -0.01241302, -0.0378418 , ...,  0.00770569,
          0.00475311,  0.03652954]], shape=(2, 1536)),
 'documents': ['[PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.',
  "[PlayStation 2] Grand Theft Auto: San Andreas (2004) - An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson."],
 'uris': None,
 'included': ['embeddings', 'metadatas', 'documents'],
 'data': None,
 'metadatas': [{'Description': 'A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.',
   'Name': 'Gran Turismo',
   'Publisher': 'Sony Computer Entertainment',
   'YearOfRelease': 1997,
   'Platform': 'PlayStation 1',
   'Genre': 'Racing'},
  {'Descriptio